In [ ]:
!pip install numpy mne osl-dynamics tensorflow ipyevents ipywidgets pyvista pyvistaqt pandas seaborn nibabel nilearn scikit-image numba pyyaml scikit-learn

In [2]:
import sys
!conda run -n hmmnet pip install h5io

In [3]:
import mne
import numpy as np
import scipy.io
import h5py

from osl_dynamics import analysis, inference, files
from osl_dynamics.meeg import preproc, rhino, source_recon, parcellation
from osl_dynamics.data import Data
from osl_dynamics.utils import plotting
from osl_dynamics.utils.filenames import OSLFilenames


In [ ]:
import h5py
import numpy as np
import mne
import matplotlib.pyplot as plt
from scipy.signal import welch

# =============================================================================
# 1. LOAD DATA FROM MAT FILE
# =============================================================================

subject_folder = '\\rdp.arc.ucl.ac.uk\ritd-ag-project-rd01pz-dbush99\Katja Button Press Data\s05\'
subject_filename = 'Subject5_spm_mne'

filepath = subject_folder + subject_filename + '.mat'

with h5py.File(filepath, 'r') as f:
    fid_pnt    = np.array(f['fid']['fid']['pnt']).T
    hsp_pnt    = np.array(f['fid']['pnt']).T
    data       = np.array(f['data']).T        # (n_channels, n_times)
    sfreq      = float(np.array(f['fs']).flatten()[0])
    chanpos    = np.array(f['sens']['chanpos']).T
    chanori    = np.array(f['sens']['chanori']).T
    chan_names = []
    for ref in f['chan_names'][:, 0]:
        name = ''.join(chr(c) for c in f[ref][:].flatten())
        chan_names.append(name)

print(f"Data shape: {data.shape}")
print(f"N channels: {len(chan_names)}")
print(f"Sampling rate: {sfreq} Hz")


Data shape: (343, 592450)
N channels: 343
Sampling rate: 600.0 Hz


In [ ]:
# =============================================================================
# 2. BUILD MNE RAW OBJECT
# =============================================================================
# Load CTF file for correct channel info (positions, coil types, dev_head_t)
raw_ctf = mne.io.read_raw_ctf(
    '/home/yuval/PhD/Years3-4/Scz_Cn_MEG_proj/Controls/Raw_Data/Control 4/mg04944_Fanfan_20170510_01.ds',
    preload=False
)
raw_ctf_meg = raw_ctf.copy().pick_types(meg=True, ref_meg=False)
print(f"CTF MEG channels: {len(raw_ctf_meg.ch_names)}")

# Strip CTF serial number suffix
ctf_ch_names_stripped = [ch.split('-')[0] for ch in raw_ctf_meg.ch_names]

# Reorder SPM data to match CTF channel order and convert fT → T
spm_to_ctf_idx = [chan_names.index(ch) for ch in ctf_ch_names_stripped]
data_meg = data[spm_to_ctf_idx, :] / 1e15   # (273, n_times) in Tesla

# Reorder chanpos to match CTF channel order
chanpos_meg = chanpos[spm_to_ctf_idx, :]

print(f"data_meg shape: {data_meg.shape}")
print(f"data_meg std: {data_meg.std():.2e}")  # should be ~1e-13

# Create Raw using CTF info (has correct positions, coil types, dev_head_t)
raw = mne.io.RawArray(data_meg, raw_ctf_meg.info)
print(f"dev_head_t: {raw.info['dev_head_t']}")

ds directory : /home/yuval/PhD/Years3-4/Scz_Cn_MEG_proj/Controls/Raw_Data/Control 4/mg04944_Fanfan_20170510_01.ds
    res4 data read.
    hc data read.
    Separate EEG position data file read.
    Quaternion matching (desired vs. transformed):
       2.66   70.12    0.00 mm <->    2.66   70.12   -0.00 mm (orig :  -52.67   46.64 -258.71 mm) diff =    0.000 mm
      -2.66  -70.12    0.00 mm <->   -2.66  -70.12   -0.00 mm (orig :   53.96  -44.38 -252.24 mm) diff =    0.000 mm
      79.15    0.00    0.00 mm <->   79.15    0.00    0.00 mm (orig :   49.71   62.60 -264.32 mm) diff =    0.000 mm
    Coordinate transformations established.
    Polhemus data for 3 HPI coils added
    Device coordinate locations for 3 HPI coils added
    Measurement info composed.
Finding samples for /home/yuval/PhD/Years3-4/Scz_Cn_MEG_proj/Controls/Raw_Data/Control 4/mg04944_Fanfan_20170510_01.ds/mg04944_Fanfan_20170510_01.meg4: 
    System clock channel is available, checking which samples are valid.
    1 x 5

In [ ]:
# =============================================================================
# 3. SAVE
# =============================================================================
import os

os.chdir(subject_folder)        
outfile = subject_folder + subject_filename + '.fif'
raw.save(outfile, overwrite=True)
print("Saved!")


Sampling frequency of the instance is already 250.0, returning unmodified.
Sfreq after resample: 250.0
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 45 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 1.00, 45.00 Hz: -6.02, -6.02 dB


Bad segment detection
---------------------
Omitting 7000 of 246854 (2.84%) samples, retaining 239854 (97.16%) samples.
Modality: mag
Mode: None
Metric: std
Significance level: 0.05
Maximum fraction: 0.1
Found 0 bad segments: 0.0/987.4 seconds rejected (0.0%)

Bad segment detection
---------------------
Omitting 7000 of 246854 (2.84%) samples, retaining 239854 (97.16%) samples.
Modality: mag
Mode: diff
Metric: std
Significance level: 0.05
Maximum fraction: 0.1
Found 0 bad segments: 0.0/987.4 seconds rejected (0.0%)
Overwriting existing file.
Writing /home/yuval/Canonical-HMM-N